# Michigan Traders — Module 7
# Statistical Signal Research  ·  *interactive workbook*

**Series:** MAT Education · Evaluation & Research
**Level:** Intermediate (builds on Modules 1–6)
**Format:** **Guided + Your-Turn**, with self-checking exercises you can run on your own laptop.

---

Module 5 gave you signals. Module 6 gave you the tools to judge them. Neither told you where a signal should come from in the first place, and "try moving averages until something works" is exactly the procedure Module 6 showed manufactures fake results.

This module supplies the missing step: **test a property of the data first, then build the signal that exploits it**.

The property we test is **mean reversion** — does this series get pulled back toward a level, or does it wander freely? That single question decides which family of strategy can possibly work:

| If the series is… | then… | and the strategy is… |
|---|---|---|
| **mean-reverting** (stationary) | deviations from the mean tend to shrink | fade the extremes |
| **a random walk** | deviations persist; there is no anchor | mean reversion cannot work |
| **trending** | deviations tend to grow | follow the move |

Individual stock prices are very close to random walks, which is why fading a single stock's price is a losing game. The trick that makes mean reversion work in equities is to construct a *combination* of prices that is stationary even though each price is not. That combination is called a **cointegrated spread**, and building one is the second half of this notebook.

## How to use this notebook

| Cell type | Where it runs | What to do |
|---|---|---|
| 🟢 **Local cell** | your laptop's Jupyter | Run it. Output is baked in so you can read along. |
| 🔵 **QC cell** | QuantConnect (LEAN) | Copy into a research notebook or algorithm. It will *not* run locally. |

**New dependency.** This module uses `statsmodels` for its statistical tests. It is pre-installed on QuantConnect. Locally, install it once:

```
pip install statsmodels
```

Answers are in `07_Statistical_Signal_Research_SOLUTIONS.ipynb`.

### Setup — four series with known properties

Because the whole module is about *detecting* a property, we build data where we already know the answer. That is the only way to tell whether a test works, and it is a habit worth keeping: before trusting a test on real data, run it on data whose truth you control.

| Series | Construction | True property |
|---|---|---|
| `walk` | random walk | **not** mean-reverting |
| `rev` | Ornstein–Uhlenbeck process pulled toward 50 | **is** mean-reverting |
| `MINE_A`, `MINE_B` | two miners driven by the same commodity | **cointegrated** |
| `TECH_C`, `TECH_D` | two tech names sharing a market factor | correlated, **not** cointegrated |

The `MINE_A`/`MINE_B` story matters. Module 6 insisted on an economic story before a statistical one, so here it is: two mining companies pulling the same metal out of the ground have revenues driven by the same commodity price. Their equity values can drift apart on company-specific news, but the shared driver keeps pulling them back into line. That is a *reason* to expect cointegration, and it is what separates a pair worth testing from a pair you found by scanning.

`TECH_C`/`TECH_D` are the trap: they move together day to day because both respond to the same market factor, but nothing anchors their *ratio*, so it can wander anywhere.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.precision", 4)

rng = np.random.default_rng(31)
n = 1008                                    # ~4 trading years
dates = pd.bdate_range("2020-01-02", periods=n)


def ornstein_uhlenbeck(n, theta, sd, mu, x0, rng):
    # A series pulled toward `mu` by strength `theta`, jostled by noise `sd`.
    x = np.empty(n)
    x[0] = x0
    for t in range(1, n):
        pull = theta * (mu - x[t - 1])      # toward the mean, proportional to distance
        x[t] = x[t - 1] + pull + rng.normal(0, sd)
    return x


# 1. A pure random walk: today's value plus a shock, no anchor anywhere.
walk = pd.Series(100 + np.cumsum(rng.normal(0, 1.0, n)), index=dates, name="walk")

# 2. A mean-reverting series anchored at 50.
rev = pd.Series(ornstein_uhlenbeck(n, 0.05, 1.0, 50.0, 50.0, rng), index=dates, name="rev")

print(f"walk: starts {walk.iloc[0]:.1f}, ends {walk.iloc[-1]:.1f}, "
      f"range {walk.min():.1f} to {walk.max():.1f}")
print(f"rev : starts {rev.iloc[0]:.1f}, ends {rev.iloc[-1]:.1f}, "
      f"range {rev.min():.1f} to {rev.max():.1f}")

walk: starts 99.6, ends 114.8, range 89.9 to 153.6
rev : starts 50.0, ends 46.6, range 42.1 to 59.5


The `ornstein_uhlenbeck` function is worth reading line by line, because it *is* the definition of mean reversion:

```python
pull = theta * (mu - x[t - 1])
```

The further `x` is below `mu`, the larger and more positive the pull. `theta` is how hard the elastic pulls: `theta = 0` gives a pure random walk, `theta = 1` snaps straight back to the mean every step. We used `theta = 0.05`, a gentle pull.

Now the two pairs. Both are built in **log price**, for a reason that will matter later: in logs, a constant ratio between two prices becomes a constant *difference*, which is what these tests can detect.

In [2]:
# 3. A cointegrated pair. Both miners are driven by the same underlying path,
#    and their gap is an OU process — it wanders, but always gets pulled back.
log_a = np.log(100) + np.cumsum(rng.normal(0.0006, 0.020, n))
true_gap = ornstein_uhlenbeck(n, 0.04, 0.020, 0.0, 0.0, rng)
log_b = -1.0 + 1.2 * log_a + true_gap          # TRUE hedge ratio is 1.2

MINE_A = pd.Series(np.exp(log_a), index=dates, name="MINE_A")
MINE_B = pd.Series(np.exp(log_b), index=dates, name="MINE_B")

# 4. Correlated but NOT cointegrated: a shared daily factor, but each name also
#    has its own random walk, so nothing anchors the gap between them.
factor = rng.normal(0.0005, 0.010, n)
TECH_C = pd.Series(100 * np.exp(np.cumsum(1.0 * factor + rng.normal(0, 0.004, n))),
                   index=dates, name="TECH_C")
TECH_D = pd.Series(100 * np.exp(np.cumsum(1.1 * factor + rng.normal(0, 0.004, n))),
                   index=dates, name="TECH_D")

prices = pd.DataFrame({"MINE_A": MINE_A, "MINE_B": MINE_B,
                       "TECH_C": TECH_C, "TECH_D": TECH_D})
prices.iloc[[0, -1]].round(2)

,MINE_A,MINE_B,TECH_C,TECH_D
2020-01-02,103.29,96.07,99.31,99.21
2023-11-13,222.11,240.34,156.45,197.51


Note the line `log_b = -1.0 + 1.2 * log_a + true_gap`. The number **1.2** is the true hedge ratio, and the entire second half of this notebook is an exercise in recovering it from the prices alone. Keep it in mind — being able to check an estimate against a known truth is why we simulated rather than downloaded.

## Part 1 — Is this series mean-reverting?

## 1. What stationarity actually means

A series is **stationary** if its statistical behaviour does not depend on *when* you look at it. Formally, three things stay constant over time:

1. the **mean**
2. the **variance**
3. the **autocovariance** at each lag (how today relates to *k* days ago)

The practical consequence is the only part you need to hold on to:

> 🧠 **A stationary series has a mean to come back to. A non-stationary one does not.**

This is not a technicality. Every mean-reversion strategy says "it is far from normal, so it will come back". If there is no "normal" — if the series has no fixed mean — then "far from normal" is an undefined quantity and the strategy is betting on nothing.

Look at what that difference does to the two series we built.

In [3]:
# Split each series into four consecutive year-long chunks and compare.
def chunk_stats(s, k=4):
    parts = np.array_split(np.arange(len(s)), k)
    return pd.DataFrame({
        "mean": [s.iloc[p].mean() for p in parts],
        "std": [s.iloc[p].std() for p in parts],
    }, index=[f"year {i + 1}" for i in range(k)])


print("walk (random walk)");  print(chunk_stats(walk).round(2))
print()
print("rev (mean-reverting)"); print(chunk_stats(rev).round(2))

walk (random walk)
          mean   std
year 1   97.92  4.27
year 2  120.54  7.72
year 3  144.14  4.36
year 4  134.40  7.26

rev (mean-reverting)
         mean   std
year 1  50.90  2.76
year 2  49.33  3.07
year 3  48.78  3.44
year 4  49.93  3.47


There it is, without a single statistical test. The random walk's mean wanders from year to year — whatever level it happens to reach, it stays near, and there is no force returning it. The mean-reverting series posts essentially the same mean every year, because 50 is a real anchor.

That is the whole idea. Everything below is a way of making this judgement precisely rather than by eye.

### The trap this prevents

Here is what happens if you skip the test and z-score a random walk anyway — the mistake this module exists to prevent.

In [4]:
# "It's 2 standard deviations below its mean, so buy it."
# Let's check whether that is true, on both series. For every day the series
# looked extreme, measure what it actually did over the NEXT 20 days.
def extremes_pay(series, name, lookback=60, horizon=20):
    z = (series - series.rolling(lookback).mean()) / series.rolling(lookback).std()
    forward = series.shift(-horizon) - series          # what happened next

    low = forward[z < -2].dropna()                     # days that looked "cheap"
    high = forward[z > 2].dropna()                     # days that looked "expensive"
    print(f"{name}")
    print(f"   after z < -2 ({len(low):>3} days): next {horizon}d move {low.mean():+.2f}")
    print(f"   after z > +2 ({len(high):>3} days): next {horizon}d move {high.mean():+.2f}")
    print(f"   baseline (all days)      : next {horizon}d move "
          f"{forward.dropna().mean():+.2f}")


extremes_pay(rev, "rev (mean-reverting)")
print()
extremes_pay(walk, "walk (random walk)")

rev (mean-reverting)
   after z < -2 ( 33 days): next 20d move +1.24
   after z > +2 ( 61 days): next 20d move -3.95
   baseline (all days)      : next 20d move -0.05

walk (random walk)
   after z < -2 ( 54 days): next 20d move -1.30
   after z > +2 ( 98 days): next 20d move +0.48
   baseline (all days)      : next 20d move +0.33


Read the two blocks against each other, because this single comparison is the reason the module exists.

On `rev`, buying the dips works exactly as advertised: after an extreme low the series rises over the next month, after an extreme high it falls, and both moves are far larger than the baseline drift. The signal has real information.

On `walk`, the sign is **backwards**. Days that looked cheap went on to fall further; days that looked expensive kept rising. There was no anchor to revert to, so "far below the recent mean" was simply a description of a downtrend in progress — and fading it meant standing in front of it.

That is the whole trap, and note what it is *not*. It is not that mean reversion on a random walk is a coin flip you would break even on. It is that the signal points the wrong way relative to what actually happens, and you would pay costs on every trade for the privilege.

> ⚠️ **The most expensive mistake in this module** is applying a mean-reversion signal to a series nobody tested. The z-score *computes* fine — pandas will happily divide — and it means nothing. The arithmetic never complains.

## 2. Test one: autocorrelation of the level

**Autocorrelation** at lag *k* is the correlation between the series and itself shifted *k* days. Module 4 used `.corr()` between two different series; here we correlate a series with its own past.

For a stationary series, the link to the distant past **decays** — after enough time, where it was stops telling you where it is. For a random walk, today's value *contains* every past shock, so the autocorrelation stays near 1.0 for a long time.

In [5]:
lags = [1, 5, 10, 20, 60, 120]

acf = pd.DataFrame({
    "walk": [walk.autocorr(lag=k) for k in lags],
    "rev": [rev.autocorr(lag=k) for k in lags],
}, index=[f"lag {k}" for k in lags])
acf.round(3)

,walk,rev
lag 1,0.998,0.952
lag 5,0.991,0.768
lag 10,0.983,0.568
lag 20,0.964,0.303
lag 60,0.898,-0.037
lag 120,0.819,-0.148


Read down the columns. `walk` is still 0.7-plus after 120 days: four months later it has barely forgotten where it was. `rev` decays toward zero within about 60 days, because the pull toward 50 erases history.

This is a good first look and a bad decision rule — "decays fast enough" has no threshold. It tells you which series to investigate, not which is stationary.

## 3. Test two: the AR(1) regression and half-life

This one is both a test and a *design tool*, which is why it earns the most space.

Regress each day's **change** on the **previous level**:

$$\Delta y_t = \lambda \, y_{t-1} + c + \varepsilon_t$$

The sign of λ is the answer:

| λ | meaning |
|---|---|
| λ < 0 | high values are followed by falls, low by rises → **mean-reverting** |
| λ ≈ 0 | the level says nothing about the next change → **random walk** |
| λ > 0 | moves feed on themselves → **trending / explosive** |

### Fitting a line with `np.polyfit`

`np.polyfit(x, y, 1)` fits `y ≈ slope·x + intercept` by least squares and returns `[slope, intercept]`. The `1` is the polynomial degree. It is the least-ceremony regression in NumPy, and for a single predictor it is all we need.

In [6]:
# A quick sanity check on polyfit itself, with an answer we know.
demo_x = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
demo_y = 3.0 + 2.0 * demo_x                     # intercept 3, slope 2

slope, intercept = np.polyfit(demo_x, demo_y, 1)
print(f"recovered slope {slope:.3f}, intercept {intercept:.3f}  (expected 2 and 3)")

recovered slope 2.000, intercept 3.000  (expected 2 and 3)


Now the real thing. Two alignment details matter and both are easy to get wrong:

- `y.diff()` produces `NaN` in its first slot, so we drop it.
- The predictor must be the **lagged** level, `y.shift(1)`, not `y`. Regressing a change on its own contemporaneous level is a different (and wrong) question.

In [7]:
def ar1_lambda(series):
    # Regress the daily change on the previous level; return lambda.
    y = series.dropna()
    delta = y.diff().dropna()                   # change today
    level = y.shift(1).dropna()                 # level yesterday
    level = level.loc[delta.index]              # align the two explicitly
    lam, _ = np.polyfit(level, delta, 1)
    return float(lam)


print(f"lambda for walk : {ar1_lambda(walk):+.5f}")
print(f"lambda for rev  : {ar1_lambda(rev):+.5f}")

lambda for walk : -0.00235
lambda for rev  : -0.04729


`rev` has a clearly negative λ; `walk` sits essentially at zero. The test agrees with the truth we built in.

### Half-life: turning λ into a number you can trade on

λ by itself is hard to interpret. Convert it into **half-life** — the number of days for a deviation to decay to half its size:

$$\text{half-life} = \frac{-\ln 2}{\lambda}$$

This is the single most useful number in mean-reversion research, because it sets your holding period and your lookback:

- half-life of 3 days → an intraday or few-day trade; a 60-day lookback is far too slow
- half-life of 20 days → a swing trade; a 60-day lookback is about right
- half-life of 400 days → technically mean-reverting, untradeable in practice

In [8]:
def half_life(series):
    lam = ar1_lambda(series)
    if lam >= 0:
        return np.inf                           # no reversion to speak of
    return float(-np.log(2) / lam)


print(f"half-life of rev  : {half_life(rev):.1f} days")
print(f"half-life of walk : {half_life(walk):.1f} days")
print()
print(f"we built rev with theta = 0.05, so the true half-life is "
      f"ln(2)/0.05 = {np.log(2) / 0.05:.1f} days")

half-life of rev  : 14.7 days
half-life of walk : 295.2 days

we built rev with theta = 0.05, so the true half-life is ln(2)/0.05 = 13.9 days


The estimate for `rev` lands close to the truth, which is the confirmation we simulated the data to get.

**Now look at the number for `walk`, because it is the more instructive one.** It did not come back `inf`. It came back a finite few hundred days, because λ was estimated at a very slightly negative value — not from any real reversion, but from ordinary sampling noise. On a random walk the true λ is exactly zero, and an estimate from 1,000 noisy observations will land a hair either side of it by luck.

So the half-life formula will hand you a plausible-looking number for a series with no mean reversion whatsoever. It never refuses. This is why the next section exists: you need a test that says whether λ is *significantly* below zero, not merely below zero.

> 🧠 **A half-life on its own proves nothing.** Always read it next to a significance test. A "300-day half-life" is usually just a way of writing "no reversion detected".

Note also the guard on `lam >= 0`. A positive λ means the series is actively trending, and `-ln(2)/λ` would return a *negative* half-life — a meaningless number that still looks like a real one. Returning `inf` says "never reverts", which is the honest answer.

### ✏️ Your turn — half-life from scratch

Write `my_half_life(series)` returning the half-life in days, **without** calling the `ar1_lambda` or `half_life` helpers above.

Steps:
1. `delta` = the series' daily change, with the leading `NaN` dropped.
2. `level` = the series shifted forward one day, aligned to `delta`'s index.
3. Fit `delta ≈ λ · level + c` with `np.polyfit(level, delta, 1)`.
4. Return `-ln(2) / λ`, or `np.inf` when `λ >= 0`.

Then set `hl_rev` and `hl_walk` by applying it to `rev` and `walk`.

In [ ]:
def my_half_life(series):
    y = series.dropna()
    # TODO: change today, level yesterday, aligned
    delta = ...
    level = ...

    # TODO: fit and convert
    lam, _ = ...
    ...


hl_rev = my_half_life(rev)
hl_walk = my_half_life(walk)

print(f"rev {hl_rev:.1f} days | walk {hl_walk}")

In [ ]:
_ref = half_life(rev)
assert np.isclose(float(hl_rev), _ref, rtol=1e-6), \
    f"expected a half-life near {_ref:.2f} days for rev, got {hl_rev}"
assert 8 < hl_rev < 30, "rev was built with a ~13.9 day true half-life - yours is far off"
_w = my_half_life(walk)
assert (np.isinf(_w) or _w > 200), \
    "a random walk should give either inf or an enormous half-life"
# The function must work on a series it has not seen.
_fast = pd.Series(ornstein_uhlenbeck(600, 0.20, 1.0, 10.0, 10.0,
                                     np.random.default_rng(5)))
_fh = my_half_life(_fast)
assert 1.5 < _fh < 8, (
    f"on a theta=0.20 series the half-life should be near {np.log(2) / 0.20:.1f} days, "
    f"got {_fh:.2f} - make sure you regress the CHANGE on the LAGGED level")
print(f"✅ Correct!  rev reverts with a {float(hl_rev):.1f}-day half-life;",
      f"a faster theta=0.20 series gives {_fh:.1f} days, as it should")

## 4. Test three: the Augmented Dickey-Fuller test

The AR(1) regression gives a number. The **ADF test** gives that number a *significance*: is λ meaningfully below zero, or could this much apparent reversion happen by chance?

### What a hypothesis test is doing

If you have not met p-values before, here is the whole idea in four lines. It is worth reading carefully, because the ADF test is easy to interpret backwards.

1. Assume the boring explanation is true. That assumption is the **null hypothesis**.
2. Ask: if it were true, how often would I see data at least this extreme?
3. That probability is the **p-value**.
4. A small p-value means the boring explanation struggles to account for what you saw, so you **reject** it.

For the ADF test specifically:

> **Null hypothesis: the series has a unit root, i.e. it is NOT stationary.**

So the direction is the opposite of what most people guess:

| p-value | conclusion |
|---|---|
| **small** (< 0.05) | reject the null → evidence the series **is stationary** |
| **large** (> 0.05) | fail to reject → **no evidence** of stationarity |

And note the careful wording on the second row. Failing to reject is not proof of a random walk; it means the test could not tell. Absence of evidence is not evidence of absence, and with short samples the ADF test is genuinely weak.

In [11]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(rev)
stat, pvalue, used_lag, nobs, crit_values, _ = result

print(f"ADF statistic  : {stat:.4f}")
print(f"p-value        : {pvalue:.6f}")
print(f"lags used      : {used_lag}")
print(f"observations   : {nobs}")
print("critical values:")
for level, cv in crit_values.items():
    print(f"   {level:>4}: {cv:.3f}")

ADF statistic  : -4.9089
p-value        : 0.000034
lags used      : 0
observations   : 1007
critical values:
     1%: -3.437
     5%: -2.864
    10%: -2.568


`adfuller` returns a 6-tuple; the first two entries carry the verdict.

The **critical values** are the other way to read the same result: the test statistic is more negative than the 1% critical value, so we reject the null at the 1% level. More negative = stronger evidence of stationarity. Reporting the statistic against critical values and the p-value says the same thing twice, which is why most write-ups just quote the p-value.

Run it on both series side by side.

In [12]:
def adf_report(series, name):
    stat, pvalue = adfuller(series)[:2]
    verdict = "STATIONARY" if pvalue < 0.05 else "cannot reject a unit root"
    return {"series": name, "adf_stat": round(stat, 3),
            "p_value": round(pvalue, 5), "verdict": verdict}


pd.DataFrame([adf_report(walk, "walk"), adf_report(rev, "rev")])

,series,adf_stat,p_value,verdict
0,walk,-1.335,6.1307e-01,cannot reject a unit root
1,rev,-4.909,3.0000e-05,STATIONARY


Both verdicts match the truth we built in. The test works — on 1000 observations of a clean process.

> ⚠️ **The ADF test is weaker than it looks.** With a few hundred noisy observations it frequently fails to reject the null for series that genuinely do revert. Treat `p < 0.05` as encouraging rather than decisive, and always read it next to the half-life: a "significant" result with a 300-day half-life is not a trading opportunity.

### ✏️ Your turn — testing a series end to end

Write `is_mean_reverting(series, alpha=0.05)` that returns a **dict** with exactly these four keys:

| key | value |
|---|---|
| `adf_p` | the ADF p-value, a float |
| `half_life` | the half-life in days (use your `my_half_life`) |
| `stationary` | a bool: is `adf_p < alpha`? |
| `tradeable` | a bool: **stationary AND** half-life below 60 days |

Then apply it to `rev` as `verdict_rev` and to `walk` as `verdict_walk`.

The `tradeable` flag is the point of the exercise: statistical significance alone does not make a signal you can act on.

In [ ]:
def is_mean_reverting(series, alpha=0.05):
    # TODO: p-value from adfuller, half-life from your function
    adf_p = ...
    hl = ...

    return {
        "adf_p": ...,
        "half_life": ...,
        "stationary": ...,
        "tradeable": ...,
    }


verdict_rev = is_mean_reverting(rev)
verdict_walk = is_mean_reverting(walk)

print("rev :", verdict_rev)
print("walk:", verdict_walk)

In [ ]:
KEYS = {"adf_p", "half_life", "stationary", "tradeable"}
assert set(verdict_rev) == KEYS, f"the dict must have exactly the keys {sorted(KEYS)}"
assert np.isclose(verdict_rev["adf_p"], float(adfuller(rev)[1])), "adf_p for rev is off"
assert np.isclose(verdict_walk["adf_p"], float(adfuller(walk)[1])), "adf_p for walk is off"
assert verdict_rev["stationary"] is True or verdict_rev["stationary"] == True, \
    "rev should test as stationary"
assert not verdict_walk["stationary"], "walk should NOT test as stationary"
assert verdict_rev["tradeable"], "rev is stationary with a short half-life, so it is tradeable"
assert not verdict_walk["tradeable"], "walk is not tradeable"
# A stationary but far-too-slow series must be rejected as untradeable.
_slow = pd.Series(ornstein_uhlenbeck(8000, 0.006, 1.0, 0.0, 0.0,
                                     np.random.default_rng(9)))
_v = is_mean_reverting(_slow)
assert _v["stationary"], "sanity: the slow test series really is stationary"
assert not _v["tradeable"], (
    f"a series that is statistically stationary (p={_v['adf_p']:.5f}) but has a "
    f"{_v['half_life']:.0f} day half-life must come back stationary=True, tradeable=False - "
    "check that you test the half-life too")
print(f"✅ Correct!  rev: p={verdict_rev['adf_p']:.5f}, "
      f"half-life {verdict_rev['half_life']:.1f}d, tradeable.",
      f"A series with p={_v['adf_p']:.5f} and a {_v['half_life']:.0f}-day half-life",
      "passes the ADF test and is still untradeable.")

## 5. Z-scores: turning a stationary series into a signal

Once a series is known to be stationary, the **z-score** converts "how far from normal is it?" into a comparable number:

$$z_t = \frac{y_t - \text{mean}}{\text{standard deviation}}$$

A z of −2 means two standard deviations below the mean. Because it is unit-free, the same thresholds work across different spreads.

The only real decision is which mean and standard deviation to use, and it is the decision that quietly determines whether your backtest is honest.

In [15]:
# The WRONG way: full-sample statistics.
z_cheat = (rev - rev.mean()) / rev.std()

# The RIGHT way: a rolling window, using only data available at the time.
lookback = 60
z_honest = (rev - rev.rolling(lookback).mean()) / rev.rolling(lookback).std()

comparison = pd.DataFrame({"z_cheat": z_cheat, "z_honest": z_honest}).dropna()
print(f"correlation between the two: {comparison['z_cheat'].corr(comparison['z_honest']):.3f}")
comparison.iloc[[0, 250, 500, 750, -1]].round(3)

correlation between the two: 0.771


,z_cheat,z_honest
2020-03-25,-0.538,-2.256
2021-03-10,0.987,0.786
2022-02-23,-0.738,0.678
2023-02-08,0.694,-0.741
2023-11-13,-0.967,-0.843


`rev.mean()` uses **every** observation, including all the future ones. On 2020-06-01 it is using 2023 data. That is textbook look-ahead bias, the error Module 5 introduced and Module 6 measured, and it is easier to commit here than anywhere else in the curriculum because the line looks so innocent.

The rolling version at each date uses only the previous 60 days. It is what you could actually have computed at the time.

### Choosing the lookback

Module 3's half-life gives the answer, and this is where that number earns its keep. The window should be a small multiple of the half-life — long enough for a stable mean, short enough to adapt.

A common default is **3× the half-life**, floored at about 20 days for stability.

In [16]:
hl = half_life(rev)
suggested = max(20, int(round(3 * hl)))

print(f"half-life {hl:.1f} days -> suggested lookback {suggested} days")
print()
for lb in [10, 20, 60, 120, 250]:
    z = (rev - rev.rolling(lb).mean()) / rev.rolling(lb).std()
    print(f"  lookback {lb:>3}: {(z.abs() > 2).sum():>3} days beyond |z| > 2, "
          f"z std {z.std():.3f}")

half-life 14.7 days -> suggested lookback 44 days

  lookback  10:  47 days beyond |z| > 2, z std 1.187
  lookback  20: 101 days beyond |z| > 2, z std 1.272
  lookback  60:  97 days beyond |z| > 2, z std 1.257
  lookback 120:  57 days beyond |z| > 2, z std 1.110
  lookback 250:  61 days beyond |z| > 2, z std 1.106


The signal count is **not** monotonic in the lookback, and the shape of that column is worth a minute.

The shortest window produces the *fewest* extreme days, which surprises most people. The reason is that a 10-day mean on a series with a 14-day half-life is chasing the series itself: by the time the price has moved, the rolling mean has already followed it there, so the deviation between them stays small. You have standardised the series against a benchmark that moves with it, and the extremes disappear.

At the long end the count falls again, for the opposite reason: a 250-day window is close to the full-sample mean, stable but slow to reflect anything.

The middle of the range — a few multiples of the half-life — is where the mean is stable enough to be a real reference point and current enough to be relevant. That is the reasoning behind the 3× rule of thumb, and it is a reason rather than a result of tuning.

There is no optimum to solve for here; there is a choice to make and disclose. Module 6's parameter-stability test is how you check the choice does not matter too much.

### ✏️ Your turn — a rolling z-score

Write `rolling_z(series, lookback)` returning the rolling z-score series, using only backward-looking data.

Then, on `rev` with a lookback of **40**, compute:

| variable | definition |
|---|---|
| `z40` | the rolling z-score series |
| `n_extreme` | how many days have `abs(z) > 2`, as an int |
| `pct_valid` | the share of the series that is **not** `NaN`, as a float |

The first `lookback - 1` values must be `NaN` — that is the proof you did not peek.

In [ ]:
def rolling_z(series, lookback):
    # TODO: rolling mean and rolling std, then standardize
    ...


z40 = rolling_z(rev, 40)

n_extreme = ...
pct_valid = ...

print(f"{n_extreme} extreme days | {pct_valid:.1%} of the series is valid")

In [ ]:
_z = (rev - rev.rolling(40).mean()) / rev.rolling(40).std()
assert np.allclose(z40.dropna().values, _z.dropna().values), \
    "z40 should be (series - rolling mean) / rolling std"
assert z40.iloc[:39].isna().all(), \
    "the first 39 values must be NaN - a lookback of 40 needs 40 observations"
assert not z40.iloc[39:].isna().any(), "everything from position 39 onward should be defined"
assert int(n_extreme) == int((_z.abs() > 2).sum()), "n_extreme should count abs(z) > 2"
assert np.isclose(float(pct_valid), float(_z.notna().mean())), \
    "pct_valid should be the share of non-NaN values"
# It must not be hard-coded to a 40-day window.
_z15 = rolling_z(rev, 15)
assert _z15.iloc[:14].isna().all() and not _z15.iloc[14:].isna().any(), \
    "rolling_z must respect whatever lookback it is given"
print(f"✅ Correct!  {int(n_extreme)} days beyond |z| > 2,",
      f"and the first 39 values are NaN because you refused to look ahead")

## Part 2 — Cointegration and pairs

## 6. Correlation is not cointegration

This is the distinction the whole second half rests on, and it is worth stating in its sharpest form:

- **Correlation** is about *returns moving together day to day*. It says nothing about whether the prices stay near each other.
- **Cointegration** is about *prices staying tethered over time*. It says nothing about whether they move together on any given day.

You can have either without the other. The pair that makes money is the cointegrated one, and the pair everyone finds by scanning is the correlated one.

Our two pairs demonstrate it.

In [19]:
rets = prices.pct_change().dropna()

print("QUESTION 1 - do they move together day to day?  (correlation of returns)")
print(f"   MINE_A vs MINE_B : {rets['MINE_A'].corr(rets['MINE_B']):.3f}")
print(f"   TECH_C vs TECH_D : {rets['TECH_C'].corr(rets['TECH_D']):.3f}")
print()
print("QUESTION 2 - does the GAP between them stay put?  (ADF, from section 4)")
for a, b in [("MINE_A", "MINE_B"), ("TECH_C", "TECH_D")]:
    gap = np.log(prices[b]) - np.log(prices[a])     # log gap = log of the price ratio
    p = adfuller(gap)[1]
    print(f"   {b} - {a}: ADF p = {p:.4f}   "
          f"({'STATIONARY - it comes back' if p < 0.05 else 'wanders freely'})")

QUESTION 1 - do they move together day to day?  (correlation of returns)
   MINE_A vs MINE_B : 0.753
   TECH_C vs TECH_D : 0.868

QUESTION 2 - does the GAP between them stay put?  (ADF, from section 4)
   MINE_B - MINE_A: ADF p = 0.0029   (STATIONARY - it comes back)
   TECH_D - TECH_C: ADF p = 0.9939   (wanders freely)


The two questions give **opposite rankings**, which is the entire point.

By correlation, `TECH_C`/`TECH_D` wins: 0.868 against 0.753. If you screened a universe by return correlation and took the top pairs, this is the one you would trade.

By the gap test, it is not close in the other direction. The mining gap is stationary; the tech gap cannot reject a unit root at anything approaching significance. The tech pair moves together *on any given day* and still has a gap free to wander anywhere over months — which is exactly what section 1 told you not to trade.

Both facts about the tech pair are true at once, and they are not in tension. Correlation is a statement about **returns**, which is to say about daily changes. Cointegration is a statement about **prices**, which is to say about levels. Two series can share every daily shock and still drift permanently apart, because correlation never looks at where they ended up.

> 🧠 **Screening pairs by correlation finds exactly the wrong pairs.** It is the most common error in student pairs projects, and this output is why: the highest-correlation pair here is the untradeable one. Test the spread, not the returns.

One caveat on what we just did. Subtracting the log prices assumes the right combination is one-for-one — a hedge ratio of exactly 1. We know it is really 1.2, so that test was using the wrong weights and still detected the relationship. Section 7 estimates the weights properly, and the mining p-value improves by more than an order of magnitude when it does.

## 7. The hedge ratio

To test "the prices stay tethered" we need to say precisely what combination stays put. That is the **hedge ratio** β in

$$\log(B_t) = \alpha + \beta \cdot \log(A_t) + \text{spread}_t$$

β answers: *how many dollars of A do I short for every dollar of B I own?* Estimate it by regressing one log price on the other — the same `np.polyfit` from section 3.

### Why logs

Two reasons, and they are the reason every pairs paper works in logs:

1. A regression on **log** prices makes β a ratio relationship, which is what "these two move proportionally" means. On raw prices, β would depend on the accidental scale of each stock.
2. Differences of logs are returns, which is what we ultimately trade. This makes the spread's P&L easy to compute later.

In [20]:
log_prices = np.log(prices)

beta, alpha = np.polyfit(log_prices["MINE_A"], log_prices["MINE_B"], 1)

print(f"estimated beta  : {beta:.4f}")
print(f"estimated alpha : {alpha:.4f}")
print(f"TRUE beta       : 1.2000     (we built it in)")
print(f"TRUE alpha      : -1.0000")

estimated beta  : 1.2116
estimated alpha : -1.0421
TRUE beta       : 1.2000     (we built it in)
TRUE alpha      : -1.0000


The estimate recovers the truth closely. That is the confirmation the simulation existed to provide: the method works when the property is really there.

### The same fit, with statistics attached

`np.polyfit` gives the numbers. `statsmodels` gives the numbers *plus* the standard errors and t-statistics, which tell you how well-determined β is. `sm.add_constant` adds the column of 1s that lets the model fit an intercept.

In [21]:
import statsmodels.api as sm

model = sm.OLS(log_prices["MINE_B"], sm.add_constant(log_prices["MINE_A"])).fit()

print(f"beta      : {model.params.iloc[1]:.4f}")
print(f"std error : {model.bse.iloc[1]:.4f}")
print(f"t-stat    : {model.tvalues.iloc[1]:.1f}")
print(f"R-squared : {model.rsquared:.4f}")

beta      : 1.2116
std error : 0.0094
t-stat    : 128.2
R-squared : 0.9424


A very large t-statistic and a high R² say the relationship is tightly determined.

> ⚠️ **A high R² here is not evidence of cointegration.** Regressing any two trending series on each other produces a high R² — this is the classic *spurious regression* result. `TECH_C`/`TECH_D` would also score well. The R² tells you the line fits; only testing the **residual** tells you the relationship is real.

Which is exactly the next step.

### ✏️ Your turn — the hedge ratio and the spread

Write `hedge_ratio(y, x)` that regresses `log(y)` on `log(x)` and returns the tuple `(beta, alpha)`.

Then write `spread_series(y, x)` returning the residual series

```
log(y) − (alpha + beta · log(x))
```

Apply them to get `mine_beta`, `mine_alpha` and `mine_spread` for `MINE_B` on `MINE_A`, and `tech_spread` for `TECH_D` on `TECH_C`.

Both functions take **price** series and take the logs themselves.

In [ ]:
def hedge_ratio(y, x):
    # TODO: regress log(y) on log(x), return (beta, alpha)
    ...


def spread_series(y, x):
    # TODO: the residual of that relationship
    ...


mine_beta, mine_alpha = hedge_ratio(MINE_B, MINE_A)
mine_spread = spread_series(MINE_B, MINE_A)
tech_spread = spread_series(TECH_D, TECH_C)

print(f"beta {mine_beta:.4f} | alpha {mine_alpha:.4f}")
print(f"mine spread mean {mine_spread.mean():.2e}, std {mine_spread.std():.4f}")

In [ ]:
_b, _a = np.polyfit(np.log(MINE_A), np.log(MINE_B), 1)
assert np.isclose(mine_beta, _b), \
    f"beta should be {_b:.4f} - regress log(y) ON log(x), in that order"
assert np.isclose(mine_alpha, _a), f"alpha should be {_a:.4f}"
assert 1.0 < mine_beta < 1.4, "the true hedge ratio is 1.2, so a sane estimate is near it"
_sp = np.log(MINE_B) - (_a + _b * np.log(MINE_A))
assert np.allclose(mine_spread.values, _sp.values), "mine_spread should be the regression residual"
assert abs(float(mine_spread.mean())) < 1e-8, \
    "an OLS residual has mean ~0 by construction - yours does not, so check the formula"
assert len(tech_spread) == len(TECH_D), "tech_spread should cover the whole sample"
_tb, _ta = np.polyfit(np.log(TECH_C), np.log(TECH_D), 1)
assert np.allclose(tech_spread.values, (np.log(TECH_D) - (_ta + _tb * np.log(TECH_C))).values), \
    "tech_spread must re-estimate its own beta, not reuse the mining one"
print(f"✅ Correct!  Estimated hedge ratio {mine_beta:.4f} against a true value of 1.2")

## 8. The Engle-Granger test

Now assemble the pieces into the standard two-step procedure. Everything in it is something you have already done:

1. **Regress** one log price on the other to get β. *(section 7)*
2. **Test the residual for stationarity** with ADF. *(section 4)*

If the residual is stationary, the two prices are **cointegrated**: they can each wander forever, but a specific combination of them cannot. That combination is your tradeable series.

In [24]:
def engle_granger(y, x, name=""):
    beta, alpha = np.polyfit(np.log(x), np.log(y), 1)
    resid = np.log(y) - (alpha + beta * np.log(x))
    stat, pvalue = adfuller(resid)[:2]
    hl = half_life(resid)
    return {"pair": name, "beta": round(beta, 3), "adf_p": round(pvalue, 5),
            "half_life": round(hl, 1) if np.isfinite(hl) else np.inf,
            "cointegrated": pvalue < 0.05}


pd.DataFrame([
    engle_granger(MINE_B, MINE_A, "MINE_B ~ MINE_A"),
    engle_granger(TECH_D, TECH_C, "TECH_D ~ TECH_C"),
])

,pair,beta,adf_p,half_life,cointegrated
0,MINE_B ~ MINE_A,1.212,0.0001,16.3,True
1,TECH_D ~ TECH_C,1.428,0.3272,78.2,False


The verdicts are unambiguous and they are the opposite of what daily correlation suggested.

`MINE_B ~ MINE_A` has a residual that is strongly stationary with a workable half-life. `TECH_D ~ TECH_C` — the pair with the *higher* return correlation — cannot reject a unit root. Its spread is a random walk, and everything in section 1 applies to it.

### The packaged version

`statsmodels` ships the whole procedure as `coint`, which uses critical values adjusted for the fact that β was estimated rather than known. Use it for the verdict, and keep your own version for β, the residual and the half-life, which `coint` does not return.

In [25]:
from statsmodels.tsa.stattools import coint

for label, y, x in [("MINE", MINE_B, MINE_A), ("TECH", TECH_D, TECH_C)]:
    stat, pvalue, crit = coint(np.log(y), np.log(x))
    print(f"{label}: coint stat {stat:>7.3f}   p-value {pvalue:.5f}   "
          f"{'COINTEGRATED' if pvalue < 0.05 else 'not cointegrated'}")

MINE: coint stat  -4.667   p-value 0.00064   COINTEGRATED
TECH: coint stat  -1.912   p-value 0.57405   not cointegrated


One asymmetry worth knowing: `coint(y, x)` and `coint(x, y)` can give slightly different p-values, because the regression is not symmetric — the errors are assumed to sit in the dependent variable. In practice, run it both ways and take the less favourable answer.

### ✏️ Your turn — screening a set of pairs

Write `screen_pairs(price_df)` that tests **every unordered pair** of columns and returns a DataFrame with one row per pair and exactly these columns, in this order:

`pair`, `ret_corr`, `beta`, `adf_p`, `cointegrated`

- `pair` — the string `"Y~X"` for the two column names, in the order they were tested
- `ret_corr` — correlation of the two columns' daily percentage returns
- `beta`, `adf_p` — from the Engle-Granger procedure regressing the **second** column on the **first**
- `cointegrated` — bool, `adf_p < 0.05`

For a 4-column frame that is 6 rows. Iterate columns in their existing order, taking `x = columns[i]` and `y = columns[j]` for every `j > i`, so the pair string is `f"{y}~{x}"`. Sort the result by `adf_p` ascending.

In [ ]:
def screen_pairs(price_df):
    cols = list(price_df.columns)
    rows = []

    # TODO: loop over every unordered pair (i < j)
    ...

    return pd.DataFrame(rows).sort_values("adf_p").reset_index(drop=True)


screen_pairs(prices)

In [ ]:
out = screen_pairs(prices)
assert list(out.columns) == ["pair", "ret_corr", "beta", "adf_p", "cointegrated"], \
    f"columns must be in the stated order, got {list(out.columns)}"
assert len(out) == 6, f"4 columns give 6 unordered pairs, got {len(out)}"
assert out["adf_p"].is_monotonic_increasing, "the result must be sorted by adf_p ascending"
assert out.iloc[0]["pair"] == "MINE_B~MINE_A", \
    f"the most cointegrated pair should rank first, got {out.iloc[0]['pair']}"
assert bool(out.iloc[0]["cointegrated"]), "MINE_B~MINE_A should test as cointegrated"
_tech = out[out["pair"] == "TECH_D~TECH_C"].iloc[0]
assert not bool(_tech["cointegrated"]), "TECH_D~TECH_C should NOT test as cointegrated"
assert _tech["ret_corr"] > out.iloc[0]["ret_corr"], (
    "sanity: the tech pair has the HIGHER return correlation and is still not cointegrated - "
    "if this fails, check how you computed ret_corr")
print("✅ Correct!  The winner by cointegration is",
      f"{out.iloc[0]['pair']} (p={out.iloc[0]['adf_p']:.5f}), while the most",
      f"correlated pair {_tech['pair']} (corr {_tech['ret_corr']:.3f}) fails the test.")

## 9. Trading the spread

We have a stationary series and a half-life. Now build the strategy, using the state-holding signal pattern from Module 5.

The logic:

| condition | action |
|---|---|
| z < −2 | spread is unusually **low** → **long** the spread (buy B, sell A) |
| z > +2 | spread is unusually **high** → **short** the spread (sell B, buy A) |
| \|z\| < 0.5 | spread has normalised → **flat** |
| otherwise | hold whatever you had |

That last row is what makes it a *state*, not a per-day condition, and `ffill` is what implements it.

In [28]:
LOOKBACK, ENTRY, EXIT = 60, 2.0, 0.5

# Rebuild the spread and its rolling z-score explicitly, so this cell stands alone.
BETA, ALPHA = np.polyfit(log_prices["MINE_A"], log_prices["MINE_B"], 1)
spread = log_prices["MINE_B"] - (ALPHA + BETA * log_prices["MINE_A"])
z = (spread - spread.rolling(LOOKBACK).mean()) / spread.rolling(LOOKBACK).std()

# NaN means "no instruction today"; ffill carries the last instruction forward.
raw = pd.Series(np.nan, index=z.index)
raw[z < -ENTRY] = 1.0        # long the spread
raw[z > ENTRY] = -1.0        # short the spread
raw[z.abs() < EXIT] = 0.0    # close

pair_position = raw.ffill().fillna(0.0)

print(f"days long the spread  : {int((pair_position == 1).sum())}")
print(f"days short the spread : {int((pair_position == -1).sum())}")
print(f"days flat             : {int((pair_position == 0).sum())}")
print(f"time in the market    : {(pair_position != 0).mean():.1%}")

days long the spread  : 132
days short the spread : 145
days flat             : 731
time in the market    : 27.5%


Why the entry and exit thresholds differ: exiting at |z| < 0.5 rather than at 0 avoids churning in and out every time the spread crosses its mean. It is the same reasoning as a wider stop — the gap between entry and exit is deliberate friction that keeps turnover down.

### From position to returns

Long "the spread" means: long 1 unit of B, short β units of A. In log space the spread's move is `Δlog(B) − β·Δlog(A)`, which for small daily moves is just the return difference. Dividing by `(1 + β)` normalises to **one dollar of gross exposure** — you put roughly `1/(1+β)` into the long leg and `β/(1+β)` into the short leg.

In [29]:
# Return on one dollar of gross exposure to the spread.
spread_ret = (MINE_B.pct_change() - BETA * MINE_A.pct_change()) / (1 + BETA)

pair_ret = (pair_position.shift(1) * spread_ret).dropna()

print(f"long leg  : {1 / (1 + BETA):.1%} of gross exposure in MINE_B")
print(f"short leg : {BETA / (1 + BETA):.1%} of gross exposure in MINE_A")
print()
print(f"total return : {(1 + pair_ret).prod() - 1:.2%}")
print(f"Sharpe       : {(pair_ret.mean() * 252) / (pair_ret.std() * np.sqrt(252)):.3f}")

long leg  : 45.2% of gross exposure in MINE_B
short leg : 54.8% of gross exposure in MINE_A

total return : 35.54%
Sharpe       : 0.960


Note `pair_position.shift(1)` — the same causality rule from Module 5. You compute today's z-score after today's close, so you can only hold the position from tomorrow.

### Confirming it is actually market-neutral

The whole promise of a pair trade is that it does not care where the market goes. That is a claim, and Module 6 taught the tool for checking it: correlation and beta against the thing you are supposed to be neutral to.

In [30]:
leg_ret = MINE_A.pct_change().loc[pair_ret.index]

corr_to_leg = pair_ret.corr(leg_ret)
beta_to_leg = np.cov(pair_ret, leg_ret)[0, 1] / np.var(leg_ret, ddof=1)

print(f"correlation with MINE_A : {corr_to_leg:+.3f}")
print(f"beta to MINE_A          : {beta_to_leg:+.3f}")
print()
print(f"for contrast, MINE_B's beta to MINE_A: "
      f"{np.cov(MINE_B.pct_change().dropna(), MINE_A.pct_change().dropna())[0, 1] / np.var(MINE_A.pct_change().dropna(), ddof=1):+.3f}")

correlation with MINE_A : +0.003
beta to MINE_A          : +0.001

for contrast, MINE_B's beta to MINE_A: +1.175


Beta to the underlying is essentially zero, against roughly 1.2 for simply holding `MINE_B`. The hedge did what it was supposed to do: whatever this strategy earns, it did not earn it by taking market exposure.

That is exactly the alpha-versus-beta distinction from Module 6 section 7, and it is why market-neutral strategies are valued out of proportion to their raw returns — a Sharpe of 0.9 with zero market beta is a genuinely different object from a Sharpe of 0.9 that is 55% market.

### ✏️ Your turn — build the pair strategy

Write `pair_signal(spread, lookback, entry, exit_z)` returning the position series (`+1` long spread, `−1` short, `0` flat), using the `NaN` + `ffill` state pattern above.

Then write `pair_returns(y, x, beta, positions)` returning the strategy's daily return series, computed as

```
positions.shift(1) × (y.pct_change() − beta × x.pct_change()) / (1 + beta)
```

with `NaN`s dropped.

Apply both with **lookback 40, entry 2.0, exit 0.5** on the mining pair, then set:

| variable | definition |
|---|---|
| `pos40` | the position series |
| `ret40` | the return series |
| `sharpe40` | annualized Sharpe of `ret40` |
| `beta40` | beta of `ret40` to `MINE_A`'s returns, `cov / var` with `ddof=1` |

In [ ]:
def pair_signal(spread, lookback, entry, exit_z):
    z = rolling_z(spread, lookback)
    raw = pd.Series(np.nan, index=z.index)
    # TODO: the three instructions, then carry state forward
    ...


def pair_returns(y, x, beta, positions):
    # TODO: gross-exposure-normalized spread return, lagged by one day
    ...


pos40 = pair_signal(mine_spread, 40, 2.0, 0.5)
ret40 = pair_returns(MINE_B, MINE_A, mine_beta, pos40)

sharpe40 = ...
beta40 = ...

print(f"Sharpe {sharpe40:.3f} | beta to MINE_A {beta40:+.3f} "
      f"| in market {(pos40 != 0).mean():.1%}")

In [ ]:
_z = rolling_z(mine_spread, 40)
_raw = pd.Series(np.nan, index=_z.index)
_raw[_z < -2.0] = 1.0
_raw[_z > 2.0] = -1.0
_raw[_z.abs() < 0.5] = 0.0
_pos = _raw.ffill().fillna(0.0)
_sr = (MINE_B.pct_change() - mine_beta * MINE_A.pct_change()) / (1 + mine_beta)
_ret = (_pos.shift(1) * _sr).dropna()
assert np.allclose(pos40.values, _pos.values), \
    "pos40 is off - use NaN for 'no instruction' and ffill to hold the state"
assert set(np.unique(pos40.values)) <= {-1.0, 0.0, 1.0}, "positions must be -1, 0 or +1"
assert (pos40 != 0).any(), "the strategy should take some positions"
assert np.allclose(ret40.values, _ret.values), \
    "ret40 is off - divide by (1 + beta) and shift the positions by one day"
assert np.isclose(float(sharpe40), float((_ret.mean() * 252) / (_ret.std() * np.sqrt(252)))), \
    "sharpe40 should be annualized mean over annualized std"
_lg = MINE_A.pct_change().loc[_ret.index]
assert np.isclose(float(beta40), float(np.cov(_ret, _lg)[0, 1] / np.var(_lg, ddof=1))), \
    "beta40 should be cov(strategy, leg) / var(leg)"
assert abs(float(beta40)) < 0.15, \
    "a correctly hedged pair trade should have near-zero beta to either leg"
print(f"✅ Correct!  Sharpe {float(sharpe40):.3f} with a beta of {float(beta40):+.3f}",
      "to MINE_A - the return is not market exposure in disguise")

## 10. Judging it with Module 6's tools

A strategy is not finished until it has been through the analytics. Rebuild the statistics from Module 6 and read them.

In [33]:
def quick_report(r, label):
    eq = (1 + r).cumprod()
    dd = (eq / eq.cummax() - 1).min()
    ann_vol = r.std() * np.sqrt(252)
    held = (r != 0)
    tid = (held != held.shift()).cumsum()
    trades = r[held].groupby(tid[held]).apply(lambda x: (1 + x).prod() - 1)
    wins, losses = trades[trades > 0], trades[trades < 0]
    return {
        "strategy": label,
        "total": f"{(1 + r).prod() - 1:.2%}",
        "sharpe": round(float((r.mean() * 252) / ann_vol), 3),
        "max_dd": f"{dd:.2%}",
        "trades": len(trades),
        "win_rate": f"{(trades > 0).mean():.1%}",
        "payoff": round(float(wins.mean() / abs(losses.mean())), 2) if len(losses) else np.inf,
    }


# The same strategy on a shorter z-score window, for comparison.
z40_demo = (spread - spread.rolling(40).mean()) / spread.rolling(40).std()
raw40 = pd.Series(np.nan, index=z40_demo.index)
raw40[z40_demo < -ENTRY] = 1.0
raw40[z40_demo > ENTRY] = -1.0
raw40[z40_demo.abs() < EXIT] = 0.0
ret40_demo = (raw40.ffill().fillna(0.0).shift(1) * spread_ret).dropna()

pd.DataFrame([quick_report(pair_ret, "pairs (lookback 60)"),
              quick_report(ret40_demo, "pairs (lookback 40)")])

,strategy,total,sharpe,max_dd,trades,win_rate,payoff
0,pairs (lookback 60),35.54%,0.960,-7.70%,18,88.9%,1.54
1,pairs (lookback 40),57.32%,1.264,-7.70%,26,84.6%,1.76


Now put that beside Module 6's trend follower, which won **43%** of its trades with a payoff ratio near **5.0x**.

The pair trade is the mirror image: it wins the large majority of its trades, and its winners are only modestly bigger than its losers. Both make money. They make it in completely different ways, and they fail in completely different ways:

| | trend following | mean reversion |
|---|---|---|
| win rate | low (~40%) | high (~85%) |
| payoff ratio | large | modest |
| failure mode | many small losses in choppy markets | one enormous loss when the relationship breaks |
| feels like | frustrating, then vindicating | reliable, then catastrophic |

That last row is the one to remember. A mean-reversion strategy's equity curve looks wonderful right up until the spread it is trading stops mean-reverting, and by then the position is at its largest, because the z-score kept saying "even more extreme, even better".

> ⚠️ **Mean reversion without a stop is a strategy that works until it ends you.** The z-score is a measure of how far things have gone, and it cannot distinguish "unusually far" from "permanently changed".

## 11. Four ways this goes wrong

### 11.1 The hedge ratio is not stable

β was estimated on the whole sample — which means the backtest above used a number computed with future data. That is look-ahead bias in a place most people never look for it. Check how much β actually moves.

In [34]:
window = 252
rolling_beta = pd.Series(index=log_prices.index, dtype=float)

for i in range(window, len(log_prices) + 1):
    chunk = log_prices.iloc[i - window:i]
    rolling_beta.iloc[i - 1] = np.polyfit(chunk["MINE_A"], chunk["MINE_B"], 1)[0]

rb = rolling_beta.dropna()
print(f"full-sample beta : {BETA:.3f}")
print(f"rolling 252d beta: mean {rb.mean():.3f}, std {rb.std():.3f}, "
      f"range {rb.min():.3f} to {rb.max():.3f}")

full-sample beta : 1.212
rolling 252d beta: mean 1.006, std 0.164, range 0.717 to 1.296


It moves. Not catastrophically — this is a genuinely cointegrated pair — but enough that the "right" hedge is a moving target, and enough that a live implementation must re-estimate on a rolling window rather than freeze a number computed once.

The honest backtest re-estimates β using only past data at every point. It is more work and it always looks worse, which is how you know it is the real one.

### 11.2 Cointegration is a property of a period, not of a pair

Two companies cointegrate because of an economic link: the same commodity, the same customers, one owning a stake in the other. When the link breaks — a merger, a strategy change, a new product — the spread stops reverting, permanently. No statistical test on historical data can warn you, because the break has not happened yet.

Defences: re-run the test on a rolling window and stop trading when it fails; set a hard stop at a z-score you never expect to see (say |z| > 4); and cap the position so no single spread can end you.

### 11.3 Searching pairs is Module 6's problem, multiplied

This is the big one, and it is the reason we tested `MINE_A`/`MINE_B` rather than scanning.

Testing every pair in a universe of *n* stocks means *n(n−1)/2* tests. Module 6 section 8.4 showed what searching 200 candidates does to a result. Here the numbers are far worse.

In [35]:
for n_stocks in [10, 50, 100, 500]:
    n_pairs = n_stocks * (n_stocks - 1) // 2
    false_positives = n_pairs * 0.05          # at a 5% significance level
    print(f"  {n_stocks:>3} stocks -> {n_pairs:>7,} pairs -> "
          f"~{false_positives:>8,.0f} pairs pass at p < 0.05 by chance alone")

   10 stocks ->      45 pairs -> ~       2 pairs pass at p < 0.05 by chance alone
   50 stocks ->   1,225 pairs -> ~      61 pairs pass at p < 0.05 by chance alone
  100 stocks ->   4,950 pairs -> ~     248 pairs pass at p < 0.05 by chance alone
  500 stocks -> 124,750 pairs -> ~   6,238 pairs pass at p < 0.05 by chance alone


With the S&P 500 you get over 6,000 pairs that "look cointegrated" *purely by chance*. Sorting by p-value and trading the top 20 is a procedure that returns noise with near-certainty.

Let us confirm it on data we know has no relationships at all.

In [36]:
fp_rng = np.random.default_rng(77)

# 60 completely independent random walks. No pair is cointegrated. None.
independent = pd.DataFrame({
    f"RW_{i:02d}": 100 * np.exp(np.cumsum(fp_rng.normal(0, 0.012, 500)))
    for i in range(60)
})

cols = list(independent.columns)
pvals = []
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        lx, ly = np.log(independent[cols[i]]), np.log(independent[cols[j]])
        b, a = np.polyfit(lx, ly, 1)
        pvals.append(adfuller(ly - (a + b * lx))[1])

pvals = np.array(pvals)
print(f"pairs tested            : {len(pvals):,}")
print(f"'cointegrated' at p<0.05: {(pvals < 0.05).sum()} "
      f"({(pvals < 0.05).mean():.1%})   <- expected 5.0%")
print(f"'cointegrated' at p<0.01: {(pvals < 0.01).sum()} "
      f"({(pvals < 0.01).mean():.1%})   <- expected 1.0%")
print()
print("True number of cointegrated pairs in this data: ZERO.")

pairs tested            : 1,770
'cointegrated' at p<0.05: 247 (14.0%)   <- expected 5.0%
'cointegrated' at p<0.01: 50 (2.8%)   <- expected 1.0%

True number of cointegrated pairs in this data: ZERO.


Every one of those "discoveries" is false, and a scan sorted by p-value would put them at the top.

But look more carefully at the rate, because it is **worse than the 5% you were promised** — nearly three times worse. That is not a quirk of this seed. It is a real flaw in the procedure we have been using, and it has a specific cause.

### Why the do-it-yourself test over-rejects

The ADF critical values assume you are testing a series that was **handed to you**. Our residual was not handed to us: we *chose* β by least squares, and least squares chooses the β that makes the residual as small as possible. We then tested that deliberately-minimised residual for being small and well-behaved, using a table that assumed we had not gone looking.

So the test is rigged in its own favour, and the p-values it reports are too optimistic. This is exactly why `coint` exists — it uses critical values derived for the case where β was estimated from the same data.

In [37]:
# The same 1,770 pairs, judged by the properly-calibrated test.
coint_p = []
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        lx, ly = np.log(independent[cols[i]]), np.log(independent[cols[j]])
        coint_p.append(coint(ly, lx)[1])

coint_p = np.array(coint_p)

pd.DataFrame({
    "our DIY test": [f"{(pvals < a).sum()} ({(pvals < a).mean():.1%})" for a in (0.05, 0.01)],
    "statsmodels coint": [f"{(coint_p < a).sum()} ({(coint_p < a).mean():.1%})"
                          for a in (0.05, 0.01)],
    "should be": ["5.0%", "1.0%"],
}, index=["p < 0.05", "p < 0.01"])

,our DIY test,statsmodels coint,should be
p < 0.05,247 (14.0%),68 (3.8%),5.0%
p < 0.01,50 (2.8%),9 (0.5%),1.0%


`coint` lands near its nominal rate; our hand-rolled version does not.

This does not make the DIY version useless — it is how you get β, the residual and the half-life, none of which `coint` returns, and section 8's verdicts on `MINE`/`TECH` were correct and not close. But for the *decision*, use `coint`.

And notice that fixing the calibration does not rescue the scan. The properly-calibrated test still hands you dozens of "cointegrated" pairs out of data containing none, because that is what a 5% error rate over 1,770 tests *means* — roughly 88 expected false positives, no matter how good the test is. Calibration fixes the rate. It cannot fix the arithmetic of asking 1,770 questions.

> 🧠 **The defence is the economic story, and it has to come first.** Decide *before* testing which pairs have a reason to be tethered — same commodity, same supply chain, share classes of one company, a stock and its own ETF. Then test that short list. Ten hypotheses chosen for a reason beat 125,000 chosen by a loop.

### 11.4 The spread is not tradeable even when it is real

A spread can be genuinely stationary and still lose money, because:

- **half-life too long** — a 200-day half-life ties up capital for a year per trade
- **amplitude too small** — if the spread only ever moves 0.4%, costs eat it (Module 6 section 6)
- **shorting constraints** — one leg may be expensive to borrow, or impossible
- **the legs are illiquid** — the spread you can trade is not the one you backtested

Always run the Module 6 cost ladder on a pairs strategy. Pair trades change position more often than trend strategies, so they are far more cost-sensitive.

### ✏️ Your turn — false discovery rate

Using the `independent` DataFrame of 60 unrelated random walks, write `false_discovery_count(price_df, alpha)` that runs the Engle-Granger test on **every** unordered pair (regressing column `j` on column `i` for `j > i`) and returns the tuple `(n_pairs, n_significant)` as ints.

Then set:
- `fp_05`, `fp_01` — the `n_significant` counts at alpha 0.05 and 0.01
- `rate_05` — `n_significant / n_pairs` at alpha 0.05, a float

Every one of these is a false positive, since the data contains no real relationships at all. Two things are worth noticing in your answer: the rate comes out **above** alpha, for the reason just described, and tightening alpha cuts the count sharply without ever reaching zero.

In [ ]:
def false_discovery_count(price_df, alpha):
    cols = list(price_df.columns)
    n_pairs = 0
    n_significant = 0

    # TODO: every unordered pair, Engle-Granger, count p < alpha
    ...

    return n_pairs, n_significant


n_pairs, fp_05 = false_discovery_count(independent, 0.05)
_, fp_01 = false_discovery_count(independent, 0.01)

rate_05 = ...

print(f"{n_pairs} pairs | {fp_05} 'significant' at 5% ({rate_05:.1%}) "
      f"| {fp_01} at 1%")

In [ ]:
assert int(n_pairs) == 60 * 59 // 2, \
    f"60 columns give {60 * 59 // 2} unordered pairs, got {n_pairs}"
_p = []
for i in range(60):
    for j in range(i + 1, 60):
        _lx = np.log(independent[f"RW_{i:02d}"])
        _ly = np.log(independent[f"RW_{j:02d}"])
        _b, _a = np.polyfit(_lx, _ly, 1)
        _p.append(adfuller(_ly - (_a + _b * _lx))[1])
_p = np.array(_p)
assert int(fp_05) == int((_p < 0.05).sum()), \
    f"expected {(_p < 0.05).sum()} pairs under p=0.05, got {fp_05}"
assert int(fp_01) == int((_p < 0.01).sum()), \
    f"expected {(_p < 0.01).sum()} pairs under p=0.01, got {fp_01}"
assert np.isclose(float(rate_05), int(fp_05) / int(n_pairs)), \
    "rate_05 should be n_significant / n_pairs"
assert fp_05 > 0, "with 1770 tests you should find some false positives"
assert fp_01 < fp_05, "a stricter threshold must yield fewer discoveries"
print(f"✅ Correct!  {int(fp_05)} of {int(n_pairs)} pairs of UNRELATED random walks",
      f"'passed' at 5% - a {float(rate_05):.1%} rate, well above the 5% the threshold",
      f"promises - and {int(fp_01)} still passed at 1%. Every single one is false.")

## 12. On QuantConnect

Research first, in a QuantBook notebook. This is Module 4's `qb.history` feeding this module's tests.

In [ ]:
# 🔵 QC cell — pair research in the QuantConnect research environment
qb = QuantBook()

symbols = [qb.add_equity(t, Resolution.DAILY).symbol for t in ["EWA", "EWC"]]
history = qb.history(symbols, 1000, Resolution.DAILY)

# Module 4: long -> wide
closes = history["close"].unstack(level="symbol").dropna()
closes.columns = [str(c).split()[0] for c in closes.columns]

import numpy as np
from statsmodels.tsa.stattools import adfuller

log_px = np.log(closes)
beta, alpha = np.polyfit(log_px["EWA"], log_px["EWC"], 1)
resid = log_px["EWC"] - (alpha + beta * log_px["EWA"])

delta = resid.diff().dropna()
level = resid.shift(1).dropna().loc[delta.index]
lam = np.polyfit(level, delta, 1)[0]

print(f"hedge ratio : {beta:.4f}")
print(f"ADF p-value : {adfuller(resid)[1]:.5f}")
print(f"half-life   : {-np.log(2) / lam:.1f} days")
print(f"current z   : {(resid.iloc[-1] - resid.mean()) / resid.std():.2f}")

`EWA` and `EWC` are the Australia and Canada country ETFs — two commodity-driven developed economies. They are the standard textbook pair precisely because there is an economic reason to expect a link, which is the discipline section 11.3 demanded.

Now the algorithm. The structure is Module 3's five pillars, with this module's statistics in the middle.

In [ ]:
# 🔵 QC cell — a pairs trading algorithm
class PairsTrading(QCAlgorithm):

    def initialize(self):
        self.set_start_date(2018, 1, 1)
        self.set_end_date(2024, 1, 1)
        self.set_cash(100_000)
        self.set_brokerage_model(BrokerageName.INTERACTIVE_BROKERS_BROKERAGE,
                                 AccountType.MARGIN)

        self.a = self.add_equity("EWA", Resolution.DAILY).symbol
        self.c = self.add_equity("EWC", Resolution.DAILY).symbol

        self.lookback = 252      # for re-estimating the hedge ratio
        self.z_window = 60
        self.entry, self.exit_z = 2.0, 0.5
        self.state = 0           # -1 short spread, 0 flat, +1 long spread

        self.set_warm_up(self.lookback + 5, Resolution.DAILY)

    def on_data(self, data: Slice):
        if self.is_warming_up:
            return
        if not (data.bars.contains_key(self.a) and data.bars.contains_key(self.c)):
            return

        # Re-estimate on TRAILING data only - never the whole backtest.
        hist = self.history([self.a, self.c], self.lookback, Resolution.DAILY)
        if hist.empty:
            return

        closes = hist["close"].unstack(level="symbol").dropna()
        if len(closes) < self.z_window + 10:
            return

        log_px = np.log(closes)
        beta, alpha = np.polyfit(log_px[log_px.columns[0]],
                                 log_px[log_px.columns[1]], 1)
        resid = log_px[log_px.columns[1]] - (alpha + beta * log_px[log_px.columns[0]])

        recent = resid.iloc[-self.z_window:]
        z = (resid.iloc[-1] - recent.mean()) / recent.std()

        # Same state machine as the local version.
        if self.state == 0 and z < -self.entry:
            self.rebalance(beta, +1)
        elif self.state == 0 and z > self.entry:
            self.rebalance(beta, -1)
        elif self.state != 0 and abs(z) < self.exit_z:
            self.liquidate()
            self.state = 0
        elif self.state != 0 and abs(z) > 4.0:
            self.liquidate()          # the relationship may have broken
            self.state = 0
            self.debug(f"{self.time.date()} stopped out at z={z:.2f}")

    def rebalance(self, beta, direction):
        # direction +1 = long the spread (long C, short A).
        gross = 1.0 + beta
        self.set_holdings(self.c, direction * 1.0 / gross)
        self.set_holdings(self.a, -direction * beta / gross)
        self.state = direction

Three things in there are the module's lessons made executable.

`self.history([...], self.lookback, ...)` inside `on_data` returns only bars **up to now** — that is how you re-estimate β without look-ahead. Fitting it once in `initialize` on the whole period would be the bias from section 11.1.

The `abs(z) > 4.0` branch is section 11.2's hard stop: if the spread has gone somewhere it should never go, assume the relationship broke rather than that the opportunity got better.

`gross = 1.0 + beta` mirrors the local normalization, so the two legs together use about one unit of portfolio value rather than levering up — the `set_holdings` trap from Module 6 section 1.

### ✏️ Your turn — the research checklist

Write `pair_verdict(y, x, max_half_life=60, min_abs_z=1.5)` returning a dict with exactly these keys, in this order:

| key | value |
|---|---|
| `beta` | the hedge ratio |
| `adf_p` | ADF p-value of the spread |
| `half_life` | half-life of the spread in days |
| `current_z` | the **full-sample** z-score of the spread's last value |
| `verdict` | a string, decided by the rules below |

The `verdict` rules, checked in this order:

1. `"not cointegrated"` if `adf_p >= 0.05`
2. `"too slow"` if the half-life exceeds `max_half_life`
3. `"wait"` if `abs(current_z) < min_abs_z`
4. `"long spread"` if `current_z <= -min_abs_z`
5. `"short spread"` otherwise

Then set `mine_verdict` for `(MINE_B, MINE_A)` and `tech_verdict` for `(TECH_D, TECH_C)`.

In [ ]:
def pair_verdict(y, x, max_half_life=60, min_abs_z=1.5):
    beta, alpha = hedge_ratio(y, x)
    spread = spread_series(y, x)

    adf_p = ...
    hl = ...
    current_z = ...

    # TODO: apply the five rules in order
    verdict = ...

    return {"beta": beta, "adf_p": adf_p, "half_life": hl,
            "current_z": current_z, "verdict": verdict}


mine_verdict = pair_verdict(MINE_B, MINE_A)
tech_verdict = pair_verdict(TECH_D, TECH_C)

print("MINE:", mine_verdict)
print("TECH:", tech_verdict)

In [ ]:
KEYS = ["beta", "adf_p", "half_life", "current_z", "verdict"]
assert list(mine_verdict) == KEYS, f"keys must be exactly {KEYS} in order"
_sp = spread_series(MINE_B, MINE_A)
assert np.isclose(mine_verdict["adf_p"], float(adfuller(_sp)[1])), "adf_p is off"
assert np.isclose(mine_verdict["current_z"],
                  float((_sp.iloc[-1] - _sp.mean()) / _sp.std())), \
    "current_z should use the FULL-sample mean and std of the spread"
assert tech_verdict["verdict"] == "not cointegrated", \
    "the tech pair fails the ADF test, so it must be rejected before anything else is checked"
assert mine_verdict["verdict"] != "not cointegrated", "the mining pair IS cointegrated"
# Rule ordering: a cointegrated but very slow pair must be called "too slow".
_slow = pair_verdict(MINE_B, MINE_A, max_half_life=1)
assert _slow["verdict"] == "too slow", \
    "with max_half_life=1 the mining pair must be rejected as too slow"
# And the z rules must respond to the threshold.
_always = pair_verdict(MINE_B, MINE_A, min_abs_z=0.0)
assert _always["verdict"] in ("long spread", "short spread"), \
    "with min_abs_z=0 there is no 'wait' zone, so it must pick a side"
_never = pair_verdict(MINE_B, MINE_A, min_abs_z=99.0)
assert _never["verdict"] == "wait", "with min_abs_z=99 nothing is extreme enough to trade"
print(f"✅ Correct!  MINE -> {mine_verdict['verdict']} "
      f"(p={mine_verdict['adf_p']:.5f}, half-life {mine_verdict['half_life']:.1f}d, "
      f"z={mine_verdict['current_z']:+.2f});  TECH -> {tech_verdict['verdict']}")

## Cheat sheet

**Stationarity tests**

| Task | Code |
|---|---|
| Autocorrelation at lag k | `series.autocorr(lag=k)` |
| AR(1) lambda | `np.polyfit(y.shift(1).dropna().loc[d.index], d, 1)[0]` where `d = y.diff().dropna()` |
| Half-life | `-np.log(2) / lam` |
| ADF test | `from statsmodels.tsa.stattools import adfuller` → `adfuller(series)[1]` |
| ADF null hypothesis | **not** stationary — so small p means stationary |
| Fit a line | `slope, intercept = np.polyfit(x, y, 1)` |
| Regression with t-stats | `sm.OLS(y, sm.add_constant(x)).fit()` |

**Cointegration and pairs**

| Task | Code |
|---|---|
| Hedge ratio | `beta, alpha = np.polyfit(np.log(x), np.log(y), 1)` |
| Spread | `np.log(y) - (alpha + beta * np.log(x))` |
| Engle-Granger | fit the above, then `adfuller(spread)[1]` |
| Packaged test | `from statsmodels.tsa.stattools import coint` → `coint(np.log(y), np.log(x))[1]` |
| Rolling z-score | `(s - s.rolling(n).mean()) / s.rolling(n).std()` |
| Entry/exit state | `raw[z < -2] = 1; raw[z > 2] = -1; raw[z.abs() < 0.5] = 0; raw.ffill()` |
| Spread return | `(y.pct_change() - beta * x.pct_change()) / (1 + beta)` |
| Neutrality check | `np.cov(strat, leg)[0, 1] / np.var(leg, ddof=1)` ≈ 0 |
| Lookback rule of thumb | `max(20, 3 * half_life)` |
| Pairs in a universe | `n * (n - 1) // 2` — and 5% of them pass by chance |

## Stretch goals

1. **Honest re-estimation.** Rebuild the pairs backtest re-estimating β on a trailing 252-day window at every step, instead of once on the full sample. Compare the Sharpe against section 9's. The difference is the size of the look-ahead you removed.
2. **Half-life-driven lookback.** Instead of fixing the z-score lookback at 60, set it to `3 × half_life` recomputed on a rolling basis. Does adapting help, or is it one more parameter to overfit?
3. **The break.** Splice a structural break into the mining pair: after day 700, add a permanent +0.15 shift to `log_b`. Run the strategy through it. How much does it lose, and would a `|z| > 4` stop have saved you?
4. **Bonferroni.** Section 11.3 tested 1,770 pairs. The Bonferroni correction says that to keep an overall 5% error rate you should require `p < 0.05 / 1770`. Re-run `false_discovery_count` at that threshold. How many survive? Then consider what that threshold does to your ability to find anything real.
5. **On QuantConnect**, run the `PairsTrading` algorithm on EWA/EWC, then on a pair with no economic link (say `XLE` and `XLK`). Compare. The second one is what a scan would have handed you.

## What's next

**Module 8 — Machine Learning for Alpha** applies the same discipline to models that are far better at fooling you. You will build features from the indicators of Module 5, train a classifier to predict short-horizon direction, and discover that the default way of doing this — a random train/test split — leaks the future into the past and produces beautiful, worthless results. Every guard rail from Modules 6 and 7 gets used, because with enough parameters a model will fit anything you show it.

**Official docs:**
- [QuantConnect research environment](https://www.quantconnect.com/docs/v2/research-environment)
- [statsmodels: ADF test](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.adfuller.html)
- [statsmodels: cointegration test](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.coint.html)
- [Universe selection](https://www.quantconnect.com/docs/v2/writing-algorithms/universes/key-concepts)

*MAT Education · Evaluation & Research · Module 7.*